# FastMatrix Chain Research Walkthrough
This notebook validates correctness, inspects the optimized plan, and compares execution orders. Run all cells using the project environment.

In [ ]:
import platform, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fastmatrix import multiply_chain, plan_dimensions
print(platform.platform())
print(sys.version)
print('NumPy', np.__version__)

## Known Operation-Count Example
The dimensions `(10, 100, 5, 50)` demonstrate that multiplication order can materially change arithmetic work.

In [ ]:
plan = plan_dimensions((10, 100, 5, 50))
print(plan.parenthesization())
print(plan.scalar_multiplications)

## Correctness Against NumPy

In [ ]:
rng = np.random.default_rng(42)
matrices = [rng.normal(size=s) for s in [(200, 20), (20, 100), (100, 10), (10, 80)]]
actual, plan = multiply_chain(matrices, backend='numpy', dtype='float64', return_plan=True)
reference = np.linalg.multi_dot(matrices)
np.testing.assert_allclose(actual, reference, rtol=1e-12, atol=1e-12)
print(plan.parenthesization(), actual.shape)

## Repeated Timing
This is an exploratory notebook measurement. Use the CLI and the README protocol for reportable experiments.

In [ ]:
def timed(fn, repeats=10):
    fn()
    values=[]
    for _ in range(repeats):
        start=time.perf_counter_ns(); fn(); values.append((time.perf_counter_ns()-start)/1e6)
    return values
optimized = timed(lambda: multiply_chain(matrices, backend='numpy', dtype='float64'))
left = timed(lambda: matrices[0] @ matrices[1] @ matrices[2] @ matrices[3])
results = pd.DataFrame({'optimized_ms': optimized, 'left_to_right_ms': left})
results.describe()

In [ ]:
results.plot(kind='box', title='Exploratory matrix-chain timings')
plt.ylabel('Milliseconds')
plt.show()

## Interpretation Checklist
- Confirm backend, dtype, dimensions, environment, and package versions.
- Examine both runtime and numerical error.
- Do not generalize from one shape chain or one machine.
- Repeat in independent processes before making performance claims.